# CCA pipeline — interactive walkthrough

**Carbon Credit linked Adaptation** (Nigeria, Kenya, Ethiopia) at 5 arc-minutes.

This notebook walks through every stage of the CCA pipeline interactively. Each section pairs:

- a **markdown context block** — what the stage does, what it reads, what it writes, where it sits in the dependency graph;
- a **code cell** — runs the corresponding R script as a subprocess, captures output, prints the last ~30 lines.

Subprocesses are used (rather than `source()`) for the same reason the orchestrator [`code/inputs/00_fetch_all.R`](../code/inputs/00_fetch_all.R) uses them: each child script calls `quit(status = 0)` when its outputs are already on disk (skip-if-exists), which would kill the kernel if executed in-process.

**Estimated wall time on a fresh machine** (25 Mbit/s down): ~45–60 minutes, of which ~30 min is downloading external datasets. Subsequent runs are seconds (every script is idempotent — re-running skips completed stages).

**Authoritative documentation** for the design decisions:
- [`README.md`](../README.md) — pipeline overview
- [`data/inputs/MANIFEST.md`](../data/inputs/MANIFEST.md) — per-dataset source + status
- [`docs/methodology_biochar.md`](../docs/methodology_biochar.md) — biochar synthesis
- [`docs/methodology_hazards.md`](../docs/methodology_hazards.md) — hazard frequency synthesis

## 0. Prerequisites

**Kernel:** this notebook expects an **R kernel**. If JupyterLab does not list R as an option, install [IRkernel](https://github.com/IRkernel/IRkernel) once:

```bash
Rscript -e 'install.packages("IRkernel"); IRkernel::installspec(user = TRUE)'
```

Then refresh JupyterLab and select **R** as the kernel for this notebook (top-right corner).

**R packages required:**

```r
install.packages(c("terra", "geodata", "yaml", "arrow", "jsonlite", "digest", "data.table"))
```

**Working directory:** the notebook locates the repo root (`ccadapt/`) automatically by walking up from the notebook's own location. You do not need to `setwd()` manually.

In [ ]:
# Verify R version and required packages.
cat("R version:", R.version.string, "\n\n")

required <- c("terra", "geodata", "yaml", "arrow", "jsonlite", "digest", "data.table")
installed <- rownames(installed.packages())
missing <- setdiff(required, installed)
if (length(missing)) {
  cat("MISSING packages: ", paste(missing, collapse = ", "), "\n")
  cat("Install with: install.packages(c(\"", paste(missing, collapse = '","'), "\"))\n", sep = "")
} else {
  cat("All required R packages are installed.\n")
}

# Optional but recommended:
for (p in c("IRdisplay", "repr")) {
  if (!p %in% installed) cat("(optional) consider installing:", p, "\n")
}

## User inputs — set these before running

Edit the variables in the next cell to control how the pipeline runs. Defaults are safe for a first interactive run.

| Variable | Default | What it does |
|---|---|---|
| `FORCE_RERUN` | `FALSE` | If `TRUE`, every stage re-runs even when outputs exist. Equivalent to `CCA_FORCE=1`. Useful after editing parameters. |
| `MC_SAMPLES` | `50` | Monte Carlo sample count for Step 5 uncertainty. 50 is a smoke test, 200 is the default, 1000 is publication. Wall time scales linearly. |
| `STOP_ON_FAIL` | `FALSE` | If `TRUE`, the notebook aborts after the first failing stage. If `FALSE`, you see a summary at the end and continue. |
| `RUN_HEAVY` | `TRUE` | If `FALSE`, skips the heaviest optional stages (constraint masks, basalt) so you can iterate quickly on the Activity 2 / 3 path. |
| `RUN_NEB` | `TRUE` | If `FALSE`, skips Activity 3 NEB (the carbon-price × adoption sweep, ~131 COGs). |

**Country list** (Nigeria, Kenya, Ethiopia) and **resolution** (5 arc-minutes) are fixed in [`code/common/config.R`](../code/common/config.R). To process a different country set, edit `CCA_COUNTRIES` there. They are intentionally not exposed here because every fetcher and builder is tied to that scope.

In [ ]:
# ============================================================
# USER-EDITABLE INPUTS
# ============================================================

FORCE_RERUN  <- FALSE   # Set TRUE to override skip-if-exists guards
MC_SAMPLES   <- 50      # 50 = smoke; 200 = default; 1000 = publication
STOP_ON_FAIL <- FALSE   # Continue past failures and report at end
RUN_HEAVY    <- TRUE    # Set FALSE to skip basalt + constraint masks
RUN_NEB      <- TRUE    # Set FALSE to skip Activity 3 NEB sweep

# ============================================================
# Show resolved inputs back
# ============================================================
cat(sprintf(
  "FORCE_RERUN  = %s\nMC_SAMPLES   = %d\nSTOP_ON_FAIL = %s\nRUN_HEAVY    = %s\nRUN_NEB      = %s\n",
  FORCE_RERUN, MC_SAMPLES, STOP_ON_FAIL, RUN_HEAVY, RUN_NEB))

## Helper setup — locate repo root, source common config, define `run_step()`

The cell below:

1. Walks up from the notebook's working directory until it finds a folder named `ccadapt/` — that is the repo root.
2. Sources [`code/common/config.R`](../code/common/config.R) so we have `CCA_ROOT`, `CCA_PATHS`, `CCA_COUNTRIES`, `CCA_RES`.
3. Defines `run_step()` — a wrapper around `system2(Rscript, ...)` that:
   - injects `CCA_FORCE` and `CCA_MC_SAMPLES` env vars based on user inputs,
   - captures stdout + stderr, prints the last 30 lines (the bit that matters),
   - records elapsed seconds + status into a global `RUN_LOG` data.frame so the final summary cell can show what worked.

You should see `CCA_ROOT = .../ccadapt` printed below and an empty run log.

In [ ]:
# Locate the repo root by walking up from getwd() to find a folder called 'ccadapt'.
find_ccadapt_root <- function() {
  wd <- normalizePath(getwd(), mustWork = FALSE)
  while (basename(wd) != "ccadapt" && dirname(wd) != wd) wd <- dirname(wd)
  if (basename(wd) != "ccadapt") {
    stop("Could not locate the ccadapt/ root. Open this notebook from inside the repo.")
  }
  wd
}
CCA_ROOT_HINT <- find_ccadapt_root()
setwd(CCA_ROOT_HINT)  # ensures relative paths in scripts resolve
source(file.path(CCA_ROOT_HINT, "code", "common", "config.R"))
source(file.path(CCA_ROOT_HINT, "code", "common", "logging.R"))

cat("CCA_ROOT       =", CCA_ROOT, "\n")
cat("CCA_COUNTRIES  =", paste(CCA_COUNTRIES, collapse = ", "), "\n")
cat("CCA_RES        =", format(CCA_RES, digits = 6), "degrees (", CCA_RES_LABEL, ")\n")

# Initialise an empty run log; appended to by run_step().
RUN_LOG <<- data.frame(
  tag = character(), script = character(), status = character(),
  seconds = numeric(), stringsAsFactors = FALSE
)

# Wrapper that runs an R script as a subprocess with the user's force / MC env
# vars applied, captures output, prints the tail, and updates RUN_LOG.
run_step <- function(tag, script_path, tail_lines = 30) {
  if (!file.exists(script_path)) {
    msg <- sprintf("[%s] MISSING script: %s", tag, script_path)
    cat(msg, "\n")
    RUN_LOG[nrow(RUN_LOG) + 1L, ] <<- list(tag, basename(script_path), "MISSING", 0)
    if (STOP_ON_FAIL) stop(msg)
    return(invisible(NULL))
  }
  cca_banner(sprintf("%s   (%s)", tag, basename(script_path)))
  envs <- character()
  if (isTRUE(FORCE_RERUN)) envs <- c(envs, "CCA_FORCE=1")
  if (is.numeric(MC_SAMPLES) && MC_SAMPLES > 0)
    envs <- c(envs, sprintf("CCA_MC_SAMPLES=%d", as.integer(MC_SAMPLES)))
  rscript <- file.path(R.home("bin"), "Rscript")
  t0 <- Sys.time()
  out <- tryCatch(
    system2(rscript, args = shQuote(script_path), env = envs,
            stdout = TRUE, stderr = TRUE),
    error = function(e) { c(paste("ERROR:", conditionMessage(e))) }
  )
  t1 <- Sys.time()
  status_attr <- attr(out, "status")
  status <- if (is.null(status_attr) || identical(status_attr, 0L)) "OK" else "FAIL"
  cat("... showing last", tail_lines, "lines of output:\n")
  cat(paste(tail(out, tail_lines), collapse = "\n"), "\n")
  cat(sprintf("[%s] %s in %.1fs\n", tag, status,
              as.numeric(difftime(t1, t0, units = "secs"))))
  RUN_LOG[nrow(RUN_LOG) + 1L, ] <<- list(
    tag, basename(script_path), status,
    as.numeric(difftime(t1, t0, units = "secs"))
  )
  if (status == "FAIL" && STOP_ON_FAIL) stop(sprintf("%s failed; aborting.", tag))
  invisible(out)
}

INPUTS_DIR    <- file.path(CCA_ROOT, "code", "inputs")
ACTIVITY2_DIR <- file.path(CCA_ROOT, "code", "activity2")
ACTIVITY3_DIR <- file.path(CCA_ROOT, "code", "activity3")
cat("Helper setup complete. RUN_LOG is empty and ready.\n")

## Phase 1 — External fetchers

Every external dataset has a fetcher under [`code/inputs/`](../code/inputs/) that downloads, caches, validates, and writes a provenance sidecar. All are idempotent: they skip silently if outputs are already on disk. Set `FORCE_RERUN = TRUE` to bypass the skip guards.

The order below matches the orchestrator. Dependencies:

- `template` (GADM via `geodata`) is consumed by every clipping step downstream.
- `mapspam` is consumed by Step 2, the residue proxy, and the AWD fallback.
- `era` is consumed by Step 2 and the buffering coefficient builder.
- `naturebase` provides the mitigation rasters Step 1 reads directly.

Per-dataset detail (size, license, cross-check rule) is in [`data/inputs/MANIFEST.md`](../data/inputs/MANIFEST.md).

### 1.1 Naturebase NCS pathways (~500 MB)

Downloads the [Naturebase NCS](https://naturebase.org) `all_pathways.zip` bundle (16 mitigation pathway rasters at 1 km) into `data/naturebase/all_pathways/`. Step 1 reads agroforestry, cover-crop, grassland-soil-carbon and friends from this bundle directly — no resampling done at fetch time.

In [ ]:
run_step("naturebase", file.path(INPUTS_DIR, "fetch_naturebase.R"))

### 1.2 ERA derived meta-analysis

Resolves `ERA_Derived.rda` via three tiers: (1) local copy at `data/inputs/era/`, (2) sibling-atlas copy at `../solutions/data/`, (3) Zenodo search candidates printed for manual confirmation. No silent auto-download — ERA versions matter. Used by Step 2 (back-transformed yield response ratios) and the buffering-coefficient builder.

In [ ]:
run_step("era", file.path(INPUTS_DIR, "fetch_era.R"))

### 1.3 MapSPAM 2010 V2r0 (~600 MB across 3 mega-zips)

Pulls the [IFPRI MapSPAM 2010 V2r0](https://doi.org/10.7910/DVN/PRFF8V) Harvard Dataverse package — three per-variable mega-zips (harvested area, production, yield) for 21 crops × 4 technologies. The fetcher unpacks once, copies the wanted `_<CROP>_A.tif` files, CCA-clips them, writes COGs, and creates `SPAM2020_*` aliases used by downstream consumers.

**Why 2010 V2r0 not 2020 V2r0:** the 2020 package requires a Dataverse Guestbook clickthrough that blocks unattended API access. 2010 V2r0 is open-access and matches the version `atlas/atlas_exposure/` uses, so CCA numbers stay comparable.

In [ ]:
run_step("mapspam", file.path(INPUTS_DIR, "fetch_mapspam.R"))

### 1.4 GADM admin boundaries + 5-arcmin CCA template

Wraps `geodata::gadm()` for NGA / KEN / ETH at adm0 / adm1 / adm2, writes GeoPackage + rasterised admin masks + lookup CSVs, then builds the 5-arcmin CCA reference template (regional + per-country) that every downstream step snaps to. Resolution is the exact rational `5/60` to avoid floating-point grid mismatch later.

In [ ]:
run_step("template", file.path(INPUTS_DIR, "build_cca_template.R"))

### 1.5 FAOSTAT producer prices

Downloads the FAOSTAT 'Producer Prices – Annual (Normalized)' bulk zip (~12 MB), filters to NGA / KEN / ETH × the last 6 years, writes a tidy CSV consumed by Step 2 / Step 3 / NEB. Ethiopia coverage is sparse — flagged in 'What needs checking' item 3 of the README.

In [ ]:
run_step("fao_prices", file.path(INPUTS_DIR, "fetch_fao_prices.R"))

### 1.6 Constraint masks (WDPA / SRTM-slope / GHSL / GMIA) — *heavy*

Optional layers used by Step 4 only when a caller passes a non-NULL argument to `cca_mask_by_*()` (defaults are NO-OP). WDPA and GMIA are manual-drop because their endpoints rotated; SRTM-slope and GHSL fetch automatically. ~75 s on first run.

Skipped when `RUN_HEAVY = FALSE`.

In [ ]:
if (RUN_HEAVY) {
  run_step("constraints", file.path(INPUTS_DIR, "fetch_constraint_masks.R"))
} else {
  cat("Skipped (RUN_HEAVY = FALSE).\n")
}

### 1.7 Enhanced rock weathering (ERW) potential

Three-tier resolution: Calabrese 2024 Zenodo → Kanzaki 2023 Zenodo → Beerling 2020 Nature SI XLSX (country totals only, used as cross-check). No global pixel-resolution ERW raster is published yet, so this step usually completes with the SI cached and Step 1 ERW arm marked `staged_optional`.

In [ ]:
run_step("erw_pot", file.path(INPUTS_DIR, "fetch_erw_potential.R"))

### 1.8 Basalt / mafic geology — *optional*

GLiM (PANGAEA) lithology + USGS MRDS basalt occurrences. Used **only** when a caller passes `distance_km` to `cca_mask_by_proximity()` — defaults to NO-OP. ~17 s; skip with `RUN_HEAVY = FALSE` if you do not need ERW proximity masking.

In [ ]:
if (RUN_HEAVY) {
  run_step("basalt", file.path(INPUTS_DIR, "fetch_basalt.R"))
} else {
  cat("Skipped (RUN_HEAVY = FALSE).\n")
}

### 1.9 AWD rice methane (Cao 2026 → IPCC Tier-1 fallback)

Cao 2026 is anticipated; its Zenodo DOI returns false-positive matches so auto-download is intentionally OFF. The fallback always runs: builds an IPCC 2019 Tier-1 AWD methane mitigation raster from MapSPAM rice area using `EFc=1.19 kg CH4/ha/day, SFw=0.55, GWP20=80`. Method tag goes into `<tif>.method.json`.

In [ ]:
run_step("awd", file.path(INPUTS_DIR, "fetch_cao_awd.R"))

## Phase 2 — Pre-processing builders (synthesised layers)

Layers in this phase are **synthesised from earlier downloads + auditable parameter files** — they have no external HTTP source. Each writes a `.method.json` sidecar (instead of `.source.json`) recording every parameter value and cross-check result.

Order matters:
1. Suitability surfaces (biochar, ERW) — depend on SoilGrids + WorldClim auto-fetched via `geodata`.
2. Residue proxy — depends on MapSPAM production rasters.
3. **Biochar potential** — depends on residue proxy + biochar suitability. Synthesised from MapSPAM residues × pyrolysis parameters (Woolf 2010 + Lehmann & Joseph 2015 + Wang 2016). See [docs/methodology_biochar.md](../docs/methodology_biochar.md).
4. **Hazard frequencies** — depends on WorldClim climatology + Ecocrop crop optima. Produces 27 COGs (3 crops × 3 hazards × 3 severities). See [docs/methodology_hazards.md](../docs/methodology_hazards.md).
5. Buffering coefficients — depends on ERA derived meta-analysis.

### 2.1 Biochar fuzzy suitability surface

Geometric mean of trapezoid membership functions over pH, organic carbon, clay, and rainfall. Triggers `geodata::soil_world()` and `geodata::worldclim_global()` on first run (~1.2 GB total cache); subsequent runs read from the cache.

In [ ]:
run_step("biochar_suit", file.path(INPUTS_DIR, "build_biochar_suitability.R"))

### 2.2 ERW fuzzy suitability surface

Same trapezoid-combine pattern over pH, clay, temperature, and rainfall. Reuses the SoilGrids + WorldClim cache populated by 2.1.

In [ ]:
run_step("erw_suit", file.path(INPUTS_DIR, "build_erw_suitability.R"))

### 2.3 Crop-residue biomass proxy

Per-pixel sum of `MapSPAM_production × IPCC residue:product ratio` across 21 crops. Ratios are from IPCC 2019 Refinement Vol 4 Table 11.2, written to a CSV beside the raster for auditability. Output unit: tonnes dry residue / pixel / yr — consumed directly by the biochar synthesiser in 2.4.

In [ ]:
run_step("residue", file.path(INPUTS_DIR, "build_residue_proxy.R"))

### 2.4 Biochar mitigation potential (synthesised, BIOCHAR-ONLY)

Replaces the deprecated Sanderman MG-bundle download. Computes per pixel:

```
biochar [tCO2e/ha/yr] = residue [t/pixel/yr]
  × R_available × Y_biochar × C_biochar × P_stable_100yr
  × (44 / 12)
  / cellSize_ha
  × suitability(pH, OC, clay)
```

Parameters with citations live in [`code/inputs/data/biochar_pyrolysis_params.yaml`](../code/inputs/data/biochar_pyrolysis_params.yaml). Cross-checks against the Woolf 2010 SI Tab S6 country envelope (NGA 0.6–1.4 / KEN 0.3–0.9 / ETH 0.4–1.1 tCO₂e/ha/yr) are logged at INFO/WARN — never abort. Full derivation: [docs/methodology_biochar.md](../docs/methodology_biochar.md).

In [ ]:
run_step("biochar_pot", file.path(INPUTS_DIR, "build_biochar_potential.R"))

### 2.5 Hazard frequency rasters (synthesised, 27 COGs)

3 crops × 3 hazards × 3 severities = 27 per-pixel probability rasters consumed by Step 3. Synthesised from WorldClim 2.1 monthly climatology + Ecocrop crop optima + literature CVs. Two-tier resolution: probes the atlas `hazards_prototype/Data/` cache first, falls back to local synthesis. The synthesiser auto-fetches `geodata::worldclim_global("tmax", res=10)` if the cache is missing the monthly Tmax files (~36 MB). Full derivation: [docs/methodology_hazards.md](../docs/methodology_hazards.md).

In [ ]:
run_step("hazards", file.path(INPUTS_DIR, "build_hazard_frequencies.R"))

### 2.6 ERA hazard-buffering coefficients

Reads `ERA_Derived.rda`, scans `T.Descrip + C.Descrip` for hazard keywords, computes per (era_practice, hazard, severity) buffer fractions and confidence tiers. Output rows are tagged `confidence_tier = Low` until Activity-1 stakeholder review confirms (a) the ERA → CCA practice mapping rules and (b) the hazard-classification keyword scan.

In [ ]:
run_step("buffering", file.path(INPUTS_DIR, "build_buffering_coefficients.R"))

## Phase 3 — Activity 2 (Technical potential)

Five steps that produce per-pixel layers for every (country, practice) combination listed in [`code/activity2/config/practices.yaml`](../code/activity2/config/practices.yaml):

- **Step 1** — Mitigation potential M (tCO₂e / ha / yr). Reads from naturebase / synthesised biochar / staged AWD or ERW.
- **Step 2** — Production benefit P (USD / ha / yr Δyield × price). Anchored on Step 1 M > 0.
- **Step 3** — Adaptation benefit A (USD / ha / yr avoided loss). Multiplies hazard frequency × yield-loss fraction × buffer fraction × yield × price.
- **Step 4** — Integration: classification raster {Triple Win | M+P | M+A | Constrained | Non-priority} + opportunity-zone polygons.
- **Step 5** — Monte Carlo P10 / P50 / P90 over M, P, A. `MC_SAMPLES` controls the sample count.

### 3.1 Step 1 — Mitigation potential

Density-first reprojection: divides source totals by source pixel area (ha) BEFORE projecting with `method = "average"` onto the country template. Guarantees output extent == template extent and produces a properly area-weighted mean. Cross-checks Naturebase-sourced rasters against `all_pathways_tco2e_adm0.csv` at ±5 % accepted, ±15 % WARN.

In [ ]:
run_step("step1", file.path(ACTIVITY2_DIR, "step1_mitigation.R"))

### 3.2 Step 2 — Production benefit (Δyield × price)

Per crop touched by ERA evidence for the practice: `pc = exp(mean(log(MeanT/MeanC))) − 1` (back-transform of the geometric-mean response ratio). Apply pc to the MapSPAM baseline yield raster, **converted from kg/ha to t/ha on read**, then multiply by the country-specific FAOSTAT producer price. Missing prices → NA, never silent zeros.

In [ ]:
run_step("step2", file.path(ACTIVITY2_DIR, "step2_production.R"))

### 3.3 Step 3 — Adaptation benefit (avoided loss)

```
avoided_loss_usd_ha_yr = Σ_{hazard, severity, crop} [
  P(hazard, severity, crop)         # data/staging/hazards/<crop>/<hazard>/<severity>_5arcmin.tif
  × yield_loss_frac(hazard, severity, crop)   # literature table embedded in script
  × buffer_frac(practice, hazard, severity)   # ERA meta-analysis (capped at 1.0)
  × baseline_yield_t_ha(crop)        # MapSPAM Y, divided by 1000 on read
  × price_usd_tonne(crop, country)   # FAOSTAT
]
```

If a hazard frequency raster is missing for a given (crop, hazard, severity), that contribution is silently skipped — never imputed.

In [ ]:
run_step("step3", file.path(ACTIVITY2_DIR, "step3_adaptation.R"))

### 3.4 Step 4 — Integration + opportunity zones

Rank-normalise M, P, A within each country, flag at the quantile cutoff in [`thresholds.yaml`](../code/activity2/config/thresholds.yaml) (default 0.60), classify into 5 categories. Contiguous patches polygonised with `terra::patches(... directions = 8)`. Note the open issue under 'What needs checking' item 4: zone `area_ha` is inflated when the seed raster is one large patch — classification COGs are correct, only the polygon attribute is off.

In [ ]:
run_step("step4", file.path(ACTIVITY2_DIR, "step4_integration.R"))

### 3.5 Step 5 — Monte Carlo uncertainty

Per (country, practice): draw `MC_SAMPLES` log-normal multipliers with per-layer CVs (M = 0.30, P = 0.50, A = 0.60), reduce the per-pixel stack to P10 / P50 / P90 via `terra::app(..., quantile)`. Wall time scales linearly with `MC_SAMPLES` — set 50 for a smoke test, 200 for the default, 1000 for publication-grade runs.

In [ ]:
run_step("step5", file.path(ACTIVITY2_DIR, "step5_uncertainty.R"))

## Phase 4 — Activity 3 (Net Economic Benefit)

```
annual_cost     = transition × annuity_factor(T, r) + implementation + mrv × M
annual_benefit  = cp × M + A + P
NEB_ha_yr       = annual_benefit − annual_cost
NPV_ha          = NEB_ha_yr × Σ (1+r)^(−t) for t = 1..T
National_NPV    = Σ_pixels NPV_ha × cell_area_ha × adoption_frac
```

Sweeps over carbon prices `{5, 15, 40, 80}` USD/tCO₂e × adoption rates `{5, 15, 30, 50}` % at a 20-year horizon, 8 % discount rate. Produces ~131 NPV COGs + 3 admin CSVs. All knobs editable in [`code/activity3/neb_config.yaml`](../code/activity3/neb_config.yaml).

Skipped when `RUN_NEB = FALSE`.

In [ ]:
if (RUN_NEB) {
  run_step("neb", file.path(ACTIVITY3_DIR, "neb.R"))
} else {
  cat("Skipped (RUN_NEB = FALSE).\n")
}

## Phase 5 — Run summary

The final orchestrator-style summary table. `OK` means `Rscript` returned exit code 0; `FAIL` means non-zero (likely a stop() inside the script — re-run that single cell with `FORCE_RERUN = TRUE` to see the full output). `MISSING` means the script file does not exist at the expected path.

In [ ]:
print(RUN_LOG, row.names = FALSE)
cat(sprintf("\nTotal wall time: %.1f s (%.1f min)\n",
            sum(RUN_LOG$seconds), sum(RUN_LOG$seconds) / 60))
if (any(RUN_LOG$status == "FAIL")) {
  cat(sprintf("\nWARNING: %d step(s) failed — see logs above.\n",
              sum(RUN_LOG$status == "FAIL")))
} else if (any(RUN_LOG$status == "MISSING")) {
  cat(sprintf("\nNote: %d step(s) skipped because the script was not found.\n",
              sum(RUN_LOG$status == "MISSING")))
} else {
  cat("\nAll steps OK.\n")
}

## Phase 6 — Diagnostics & quick visualisation

Spot-check a handful of outputs to confirm they look right. The cells below are independent — run any individually.

### 6.1 Inspect the synthesised biochar potential layer

Reads the COG, prints summary stats, and plots a quick raster of biochar tCO₂e/ha/yr.

In [ ]:
library(terra)
biochar_tif <- file.path(CCA_PATHS$staging_root, "biochar",
                         "biochar_potential_tco2e_ha_yr.tif")
if (!file.exists(biochar_tif)) {
  cat("Biochar potential not yet built. Run phase 2.4.\n")
} else {
  r <- terra::rast(biochar_tif)
  print(r)
  cat("\nSummary statistics (cropland pixels only):\n")
  print(terra::global(r, c("min", "mean", "max"), na.rm = TRUE))
  terra::plot(r, main = "Biochar potential (tCO2e/ha/yr) — synthesised")
}

### 6.2 Inspect a hazard frequency layer

Maize × heat × severe — should peak in low-elevation hot zones (Sahel, lowland Kenya).

In [ ]:
haz_tif <- file.path(CCA_PATHS$staging_root, "hazards",
                     "maiz", "heat", "severe_5arcmin.tif")
if (!file.exists(haz_tif)) {
  cat("Hazard rasters not yet built. Run phase 2.5.\n")
} else {
  r <- terra::rast(haz_tif)
  cat("Country means:\n")
  for (iso3 in CCA_COUNTRIES) {
    adm0 <- terra::vect(cca_admin_path(iso3, "adm0"))
    cat(sprintf("  %s: %.3f\n", iso3,
                terra::global(terra::mask(r, adm0), "mean", na.rm = TRUE)[[1]]))
  }
  terra::plot(r, main = "P(maize, heat, severe) — hazard frequency")
}

### 6.3 Read provenance sidecars (`.method.json`)

Every synthesised layer carries a `.method.json` sidecar with formula, parameter values, input filenames, and cross-check results. The biochar sidecar is shown below as an example — useful when reviewing a stakeholder draft.

In [ ]:
sidecar <- file.path(CCA_PATHS$staging_root, "biochar",
                     "biochar_potential_tco2e_ha_yr.tif.method.json")
if (file.exists(sidecar)) {
  cat(paste(readLines(sidecar), collapse = "\n"), "\n")
} else {
  cat("Biochar sidecar not present yet.\n")
}

### 6.4 Step 1 mitigation admin summary

Per-(country, practice) totals in tCO₂e/yr and tCO₂e/ha/yr. Useful for cross-checking against the Naturebase published adm0 totals.

In [ ]:
for (iso3 in CCA_COUNTRIES) {
  csv <- file.path(CCA_PATHS$step1_dir, sprintf("mitigation_admin_%s.csv", iso3))
  if (!file.exists(csv)) next
  cat("\n===", iso3, "===\n")
  print(utils::read.csv(csv), row.names = FALSE)
}

### 6.5 Step 3 adaptation admin summary

Per-(practice, crop, hazard, severity) avoided-loss contribution in USD/ha/yr. Many rows by design — each is one term in the Step 3 sum so the contribution is fully auditable.

In [ ]:
for (iso3 in CCA_COUNTRIES) {
  csv <- file.path(CCA_PATHS$step3_dir, sprintf("adaptation_admin_%s.csv", iso3))
  if (!file.exists(csv)) {
    cat(sprintf("%s: no adaptation rows (price gaps or no buffering evidence)\n", iso3))
    next
  }
  df <- utils::read.csv(csv)
  cat(sprintf("\n=== %s (%d rows; head 8) ===\n", iso3, nrow(df)))
  print(head(df, 8), row.names = FALSE)
}

### 6.6 NEB national totals

Per (practice, country, carbon_price, adoption) NPV totals over the 20-year horizon.

In [ ]:
for (iso3 in CCA_COUNTRIES) {
  csv <- file.path(CCA_PATHS$activity3_dir, sprintf("neb_admin_%s.csv", iso3))
  if (!file.exists(csv)) next
  df <- utils::read.csv(csv)
  cat(sprintf("\n=== %s (%d rows; head 6) ===\n", iso3, nrow(df)))
  print(head(df, 6), row.names = FALSE)
}

## Re-running specific stages

**Force a single stage to re-run** even when its outputs exist — useful after editing a parameter file:

```r
FORCE_RERUN <- TRUE     # re-set globally
run_step("biochar_pot", file.path(INPUTS_DIR, "build_biochar_potential.R"))
FORCE_RERUN <- FALSE    # reset
```

**Bump Monte Carlo sample count for publication runs** (Step 5 only re-runs when forced; samples are baked into the output COG names so old outputs are not invalidated):

```r
MC_SAMPLES  <- 1000
FORCE_RERUN <- TRUE
run_step("step5", file.path(ACTIVITY2_DIR, "step5_uncertainty.R"))
```

**Update biochar pyrolysis parameters** (e.g. drop `R_available` from 0.50 to 0.30 after a regional residue-availability survey):

1. Edit [`code/inputs/data/biochar_pyrolysis_params.yaml`](../code/inputs/data/biochar_pyrolysis_params.yaml).
2. Re-run the biochar potential cell with `FORCE_RERUN = TRUE`.
3. Re-run Step 1 (which re-reads the staged biochar layer) and Activity 3 NEB.

**Update hazard parameters** (CV, growing season, severity bands): edit [`code/inputs/data/hazards_params.yaml`](../code/inputs/data/hazards_params.yaml), re-run cells 2.5 → 3.3 → 3.4 → 3.5 → 4.

**Add a new country** (e.g. Tanzania): edit `CCA_COUNTRIES` in [`code/common/config.R`](../code/common/config.R), re-run Phase 1.4 (template) onwards.

## Appendix — references

- **Pipeline overview:** [README.md](../README.md)
- **Per-dataset manifest:** [data/inputs/MANIFEST.md](../data/inputs/MANIFEST.md)
- **Methodology — biochar synthesis:** [docs/methodology_biochar.md](../docs/methodology_biochar.md)
- **Methodology — hazard frequency synthesis:** [docs/methodology_hazards.md](../docs/methodology_hazards.md)
- **Practice catalogue:** [code/activity2/config/practices.yaml](../code/activity2/config/practices.yaml)
- **Step 4 thresholds:** [code/activity2/config/thresholds.yaml](../code/activity2/config/thresholds.yaml)
- **NEB knobs:** [code/activity3/neb_config.yaml](../code/activity3/neb_config.yaml)
- **Deprecated scripts (kept for reproducibility of pre-2026-04-18 outputs):** [code/deprecated/](../code/deprecated/)